In [21]:
import numpy as np
file = open("hostel_bois.txt", "r")
messages = []
system_messages = 0
media_count = {}
deleted_count = {}

for line in file:
    line = line.strip()
    if line == "":
        continue
    if " - " not in line:
        continue
    timestamp, remaining = line.split(" - ", 1)
# skiping system messages
    if ": " not in remaining:
        system_messages += 1
        continue
    d, t = timestamp.split(", ")
    sender, msg = remaining.split(": ", 1)
    messages.append({
        "date": d,
        "time": t,
        "sender": sender,
        "message": msg})
# Counting media messages
    if msg == "<Media omitted>":
        if sender in media_count:
            media_count[sender] += 1
        else:
            media_count[sender] = 1
# Counting deleted messages
    elif msg == "This message was deleted":
        if sender in deleted_count:
            deleted_count[sender] += 1
        else:
            deleted_count[sender] = 1

file.close()

# Calculateing participants
participants = []
for msg in messages:
    if msg["sender"] not in participants:
        participants.append(msg["sender"])


# Calculateing number of days
days = []

for msg in messages:
    if msg["date"] not in days:
        days.append(msg["date"])

# Counting total media messages
total_media = 0

for person in media_count:
    total_media += media_count[person]

# Counting total deleted messages
total_deleted = 0

for person in deleted_count:
    total_deleted += deleted_count[person]

print(f"Successfully parsed {len(messages)} messages from {len(participants)} participants "
    f"over {len(days)} days, skipped {system_messages} system messages, {total_media} media-omitted, "
    f"{total_deleted} deleted messages.")

# FEATURE 2 - GROUP OVERVIEW

start_date = messages[0]["date"]
end_date = messages[len(messages) - 1]["date"]

# counting messages per person
message_count = {}
for msg in messages:
    sender = msg["sender"]
    if sender in message_count:
        message_count[sender] += 1
    else:
        message_count[sender] = 1

# Sorting people by highest to lowest message count
sorted_people = sorted(message_count.items(), key=lambda x: x[1], reverse=True)

print()
print("GROUP OVERVIEW")
print("-" * 60)

print("Group            : Hostel Bois 4ever")
print("Period           :", start_date, "to", end_date)
print("Total messages   :", len(messages))
print("Total days       :", len(days))
print("Participants     :", len(participants))

print()
print("MESSAGES PER PERSON")
print("-" * 60)

for person, count in sorted_people:
    percentage = (count / len(messages)) * 100
    print(f"{person:<15} {count:>5} ({percentage:.1f}%)")

# FEATURE 3 - MOST ACTIVE DAY AND HOUR
day_count = {} # counting messages for each day
for msg in messages:
    date = msg["date"]
    if date in day_count:
        day_count[date] += 1
    else:
        day_count[date] = 1

most_active_day = "" # Finding the day when text were maximun
most_day_messages = 0
for date in day_count:
    if day_count[date] > most_day_messages:
        most_day_messages = day_count[date]
        most_active_day = date

hour_count = {} # counting messages for each hour
for msg in messages:
    time = msg["time"]
    hour = time.split(":")[0]
    if hour in hour_count:
        hour_count[hour] += 1
    else:
        hour_count[hour] = 1

most_active_hour = ""  # Finding the busiest hour
most_hour_messages = 0
for hour in hour_count:
    if hour_count[hour] > most_hour_messages:
        most_hour_messages = hour_count[hour]
        most_active_hour = hour

print()

print("MOST ACTIVE DAY AND HOUR")
print("-" * 60)

print("Most active day :", most_active_day)
print("Messages        :", most_day_messages)

print()

print("Most active hour:", most_active_hour + ":00")
print("Messages        :", most_hour_messages)

# FEATURE 4 - ACTIVITY HEATMAP USING NUMPY
# createing a matrix with one row per participant and 24 columns (hours)
heatmap = np.zeros((len(participants), 24), dtype=int)

# adding values to the marix
for msg in messages:
    sender = msg["sender"]
    hour = int(msg["time"].split(":")[0])
    row = participants.index(sender) # Finding which row belongs to one of the senders
    heatmap[row][hour] += 1
print()

print("ACTIVITY HEATMAP")
print("-" * 60)
print("Hours ->")
for h in range(24):
    print(f"{h:2}", end=" ")

print()

for i in range(len(participants)):
    print(f"{participants[i]:10}", end=" ")
    max_value = np.max(heatmap[i])
    for value in heatmap[i]:
        if max_value == 0:
            symbol = " "
        else:
            ratio = value / max_value
            if ratio == 0:
                symbol = "."
            elif ratio <= 0.25:
                symbol = "░"
            elif ratio <= 0.50:
                symbol = "▒"
            elif ratio <= 0.75:
                symbol = "▓"
            else:
                symbol = "█"
        print(symbol, end="  ")

    print()

# FEATURE 5 - TOP WORDS
word_count = {}
'''reason for adding stop word is that it removes very common words so that top 10 words have
 more meaningful words from the chat.'''
stop_words = ["i", "is", "the", "and", "to", "in", "was", "how", "guys", "you","this","so","about",
              "for", "am", "a", "today", "at", "my","he","his","that","have", "just","me", "off",
              "which", "everyone", "telling", "from", "up","on"]

for msg in messages:
    text = msg["message"]
    # Skiping media and deleted messages
    if text == "<Media omitted>" or text == "This message was deleted":
        continue
    words = text.split()
    for word in words:
        word = word.lower()
        word = word.strip(".,!?;:'\"()[]{}<>-_*/\\|@#$%^&+=")  # Removeing punctuation
        # leaving behind empty words and stop words
        if word == "" or word in stop_words:
            continue
        if word in word_count:
            word_count[word] += 1
        else:
            word_count[word] = 1

# Sorting by highest frequency to lowest frequency
sorted_words = sorted(word_count.items(), key=lambda x: x[1], reverse=True)
print()
print("TOP 10 WORDS")
print("-" * 60)

for i in range(10):
    word = sorted_words[i][0]
    count = sorted_words[i][1]
    bar = "█" * (count // 5 + 1)
    print(f"{i+1:2}. {word:<15} {count:>4} {bar}")

# FEATURE 6 - AVERAGE RESPONSE TIME

from datetime import datetime
response_times = {}
response_counts = {}

for i in range(1, len(messages)):
    previous = messages[i - 1]
    current = messages[i]
    # only counting replies to someone else
    if previous["sender"] != current["sender"]:
        previous_time = datetime.strptime(previous["date"] + " " + previous["time"],
            "%d/%m/%y %H:%M")
        current_time = datetime.strptime(current["date"] + " " + current["time"],
            "%d/%m/%y %H:%M")
        gap = (current_time - previous_time).total_seconds() / 60
        sender = current["sender"]
        if sender in response_times:
            response_times[sender] += gap
            response_counts[sender] += 1
        else:
            response_times[sender] = gap
            response_counts[sender] = 1
print()
print("AVERAGE RESPONSE TIME")
print("-" * 60)
for person in participants:
    if person in response_times:
        average = response_times[person] / response_counts[person]
        print(f"{person:<15} {average:.2f} min")

# FEATURE 7 - DAYS ACTIVE PER PARTICIPANT
print()
print("DAYS ACTIVE PER PARTICIPANT")
print("-" * 60)
days_active = {}

for person in participants:
    active_days = []
    for msg in messages:
        if msg["sender"] == person:
            if msg["date"] not in active_days:
                active_days.append(msg["date"])
    days_active[person] = len(active_days)
    print(f"{person:<15} {len(active_days)} days")

# FEATURE 8 - PERSONALITY ARCHETYPE DETECTION
print()
print("PERSONALITY ARCHETYPES")
print("-" * 60)

# counting total messages per person
message_count = {}
for msg in messages:
    sender = msg["sender"]
    if sender in message_count:
        message_count[sender] += 1
    else:
        message_count[sender] = 1

# calculating average messages
average_messages = len(messages) / len(participants)

for i in range(len(participants)):
    person = participants[i]
    total = message_count[person]
    media = 0
    if person in media_count:
        media = media_count[person]
    deleted = 0
    if person in deleted_count:
        deleted = deleted_count[person]
    # night time (23:00–04:00)
    night = (heatmap[i][23] + heatmap[i][0] + heatmap[i][1] + heatmap[i][2] + heatmap[i][3] +
        heatmap[i][4])
    if total > 0:
        night_ratio = night / total
    else:
        night_ratio = 0
    # Average response time
    if person in response_times:
        avg_reply = response_times[person] / response_counts[person]
    else:
        avg_reply = 999999
    # Silent streak
    longest = 0
    current = 0
    person_days = []
    for msg in messages:
        if msg["sender"] == person:
            if msg["date"] not in person_days:
                person_days.append(msg["date"])
    for day in days:
        if day in person_days:
            current = 0
        else:
            current += 1
            if current > longest:
                longest = current
    if night_ratio >= 0.75:
        archetype = "Night Owl"
    elif avg_reply <= 5:
        archetype = "Fast Replier"
    elif media >= 30:
        archetype = "Media King"
    elif deleted >= 10:
        archetype = "Message Ninja"
    elif longest >= 7:
        archetype = "Ghost"
    elif total >= average_messages * 1.5:
        archetype = "Chatterbox"
    elif total <= average_messages * 0.5:
        archetype = "Silent Observer"
    else:
        archetype = "Regular Member"
    print(f"{person:<15} --> {archetype}")

Successfully parsed 3174 messages from 6 participants over 60 days, skipped 4 system messages, 32 media-omitted, 15 deleted messages.

GROUP OVERVIEW
------------------------------------------------------------
Group            : Hostel Bois 4ever
Period           : 01/04/24 to 30/05/24
Total messages   : 3174
Total days       : 60
Participants     : 6

MESSAGES PER PERSON
------------------------------------------------------------
Rahul             953 (30.0%)
Priya             718 (22.6%)
Neha              635 (20.0%)
Aman              490 (15.4%)
Karan             354 (11.2%)
Vikas              24 (0.8%)

MOST ACTIVE DAY AND HOUR
------------------------------------------------------------
Most active day : 04/05/24
Messages        : 76

Most active hour: 18:00
Messages        : 248

ACTIVITY HEATMAP
------------------------------------------------------------
Hours ->
 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 
Rahul      ░  ░  ░  ░  ░  ░  ░  ░  ░  ░  